<a href="https://colab.research.google.com/github/selvavignesh1402/My_LLM/blob/main/My_Tiny_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 Hugging Face’s library for accessing datasets.

In [ ]:
%pip install -q datasets

Check the environment and create a folder

In [ ]:
import torch
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"

project_folder = Path("/content/tiny-ai")
project_folder.mkdir(exist_ok=True)

print("PyTorch version:", torch.__version__)
print("Training device:", device)
print("Project folder:", project_folder)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cpu
Training device: cpu
Project folder: /content/tiny-ai


Download the full dataset

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download

project_folder = Path("/content/tiny-ai")
project_folder.mkdir(parents=True, exist_ok=True)

for filename in ["TinyStories-train.txt", "TinyStories-valid.txt"]:
    print(f"\nDownloading {filename}...")

    downloaded_path = hf_hub_download(
        repo_id="roneneldan/TinyStories",
        repo_type="dataset",
        filename=filename,
        local_dir=str(project_folder),
    )

    file_path = Path(downloaded_path)
    #give the size in bytes it is then converted to megabytes(MB)
    size_mb = file_path.stat().st_size / 1_000_000

    print(f"Saved: {file_path}")
    print(f"Size: {size_mb:.1f} MB")

print("\nFull dataset download complete.")

TinyStories-train.txt: reconstructing file:   0%|          |  0.00B / 1.92GB            

TinyStories-train.txt: downloading bytes:           |  0.00B            

Saved: /content/tiny-ai/TinyStories-train.txt
Size: 1924.3 MB



TinyStories-valid.txt: reconstructing file:   0%|          |  0.00B / 19.4MB            

TinyStories-valid.txt: downloading bytes:           |  0.00B            

Saved: /content/tiny-ai/TinyStories-valid.txt
Size: 19.4 MB

Full dataset download complete.


Preview a small amount of text

In [ ]:
from pathlib import Path

project_folder = Path("/content/tiny-ai")
train_path = project_folder / "TinyStories-train.txt"
validation_path = project_folder / "TinyStories-valid.txt"

# Check that both files exist.
for path in [train_path, validation_path]:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    print(f"{path.name}: {path.stat().st_size / 1_000_000:.1f} MB")

# Read only the beginning of the training file.
with train_path.open("r", encoding="utf-8") as file:
    preview = file.read(2000)

print("\nTRAINING TEXT PREVIEW:\n")
print(preview)

TinyStories-train.txt: 1924.3 MB
TinyStories-valid.txt: 19.4 MB

TRAINING TEXT PREVIEW:

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.
Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."
Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.
<|endoftext|>
Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.
One day

Count the story end markers

In [ ]:
def count_stories(path):
    count = 0

    with path.open("r", encoding="utf-8") as file:
        for line in file:
            count += line.count("<|endoftext|>")

    return count


print("Training story end markers:", count_stories(train_path))
print("Validation story end markers:", count_stories(validation_path))

Training story end markers: 2119718
Validation story end markers: 21989


Toenization Using Bytes

In [ ]:
VOCAB_SIZE = 256

def encode(text):
  byte_data = text.encode("utf-8")

  token_ids = list(byte_data)

  return token_ids

def decode(token_ids):
  byte_data = bytes(token_ids)

  text = byte_data.decode("utf-8",errors="replace")

  return text

In [ ]:
sam = "Man"

encoded = encode(sam)
decoded = decode(encoded)

print("Original Text: ",sam)
print("Token Id's: ",encoded)
print("Decoded Text: ",decoded)

assert decoded == sam
print("Passed!!")

Original Text:  Man
Token Id's:  [77, 97, 110]
Decoded Text:  Man
Passed!!


In [ ]:
with train_path.open("r",encoding="utf-8") as file:
  sample_text = file.read(200)

sample_tokens = encode(sample_text)

sample_tokens = encode(sample_text)

print("Original dataset text:\n")
print(sample_text)

print("\nFirst 30 token IDs:")
print(sample_tokens[:30])

print("\nNumber of characters:", len(sample_text))
print("Number of tokens:", len(sample_tokens))

assert decode(sample_tokens) == sample_text
print("\nDataset tokenizer test passed!")

Original dataset text:

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on

First 30 token IDs:
[79, 110, 101, 32, 100, 97, 121, 44, 32, 97, 32, 108, 105, 116, 116, 108, 101, 32, 103, 105, 114, 108, 32, 110, 97, 109, 101, 100, 32, 76]

Number of characters: 200
Number of tokens: 200

Dataset tokenizer test passed!


Tokenization Using BPE

In [ ]:
%pip install -q tokenizers